In [ ]:
# use home credit env
import json
from glob import glob
from tqdm import tqdm
import os
import numpy as np
import sys
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
%config InlineBackend.figure_format='retina'

random.seed(42)

PROJECT_PATH = '.'

sys.path.append(f"{PROJECT_PATH}/breaking-the-chain-intervention")

from datasets_for_intervention import ricechem_structure_processor
from datasets_for_intervention import ricechem_dataset

from datasets_for_intervention import averitec_structure_processor
from datasets_for_intervention import averitec_dataset

from datasets_for_intervention import tabfact_structure_processor
from datasets_for_intervention import tabfact_dsl_engine
from datasets_for_intervention import tabfact_dataset

In [ ]:
prediction_path = f"{PROJECT_PATH}/intervention_analysis/intervention_predictions"
dataset_name_list = ['ricechem', 'tabfact', 'averitec']
run_types_list = ['standard', 'tool','detailed', 'max_detailed']
models_name = ["llama32-1B", "qwen3-1.7B", "gemma2-2B", "falcon3-3B", "llama32-3B", "qwen3-4B", "falcon3-7B", "qwen3-8B", "llama31-8B", "qwen3-14B", "qwen3-32B", "llama31-70B", "qwen3-235B-a22B"]
    
    
all_prediction = {}

for dataset in dataset_name_list:
    dataset_path = os.path.join(prediction_path, dataset)
    all_prediction.setdefault(dataset, {})

    for run_type in run_types_list:

        run_path = os.path.join(dataset_path, run_type)
        if not os.path.exists(run_path):
            continue
        all_prediction[dataset].setdefault(run_type, {})
        files = os.listdir(run_path)

        for file in files:
            file_path = os.path.join(run_path, file)
            if not file.endswith(".json"):
                continue

            for model in models_name:
                if file.startswith(model):
                    with open(file_path, "r") as f:
                        preds = json.load(f)
                    all_prediction[dataset][run_type].setdefault(model, []).append(preds)

                    break

### Metrics to calculate (ID & OO(I)D)

In [ ]:
dataset = ricechem_dataset.RiceChemDataset(f"{PROJECT_PATH}/statics/datasets/RiceChem/data")
ricechem_tool = ricechem_structure_processor.RiceChemTool(dataset)

dataset = averitec_dataset.AVeriTeCDataset(f"{PROJECT_PATH}/statics/datasets/AVeriTeC/data")
averitec_tool = averitec_structure_processor.AVeriTeCTool(dataset)

dataset = tabfact_dataset.TabFactDataset(f"{PROJECT_PATH}/statics/datasets/TabFact/bootstrap_full.json",
                                        f"{PROJECT_PATH}/statics/datasets/TabFact/data/all_csv")
tabfact_engine = tabfact_dsl_engine.TabFactEngine()
tabfact_tool = tabfact_structure_processor.TabFactTool(tabfact_engine)

In [ ]:
def test():
    model_name = 'qwen3-8B'
    
    input_sample = all_prediction['ricechem']['standard'][model_name][0]['result'][1]['structure_intervention']['Correction'][0]
    meta = {"task_idx": input_sample['task_idx']}
    ricechem_rubric ={"rubric": input_sample['mediator_rubric']}
    print('RiceChem: ', ricechem_tool.calculate_score(ricechem_rubric, meta))
    
    input_sample = all_prediction['averitec']['standard'][model_name][0]['result'][1]['structure_intervention']['Local Edits'][0]
    meta = {"gold_rubric": input_sample['gold_rubric'], "gold_target": input_sample['gold_target']}
    averitec_rubric ={"rubric": input_sample['mediator_rubric']}
    print('Averitec: ', averitec_tool.calculate_score(averitec_rubric, meta))
    
    input_sample = all_prediction['tabfact']['standard'][model_name][0]['result'][0]['structure_intervention']['Correction'][0]
    meta = {"table_html_csv": input_sample['table_html_csv']}
    args ={"query": input_sample['gold_query']}
    print('TabFact: ', tabfact_tool.calculate_score(args, meta))

test()

In [ ]:
DATASET_TO_TOOL = {
    "ricechem": ricechem_tool.calculate_score,
    "averitec": averitec_tool.calculate_score,
    "tabfact": tabfact_tool.calculate_score,
}

DATASET_KEY_MAP = {
    "ricechem": {
        "y_pred_origin": "score_before_intervention",
        "mediator": "mediator_rubric",
        "y_gold": "gold_rubric",
        "y_pred_intervention": "score_after_intervention"
    },
    "tabfact": {
        "y_pred_origin": "target_before_intervention",
        "mediator": "mediator_query",
        "y_gold": "gold_query",
        "y_pred_intervention": "target_after_intervention"
    },
    "averitec": {
        "y_pred_origin": "target_before_intervention",
        "mediator": "mediator_rubric",
        "y_gold": "gold_rubric",
        "y_pred_intervention": "target_after_intervention"
    }
}


def extract_values_from_dataset_sample(sample, dataset_name, intervention=False):

    mapping = DATASET_KEY_MAP[dataset_name]
    
    y_pred_origin = sample[mapping["y_pred_origin"]] 
    mediator = sample[mapping["mediator"]]
    y_gold = sample[mapping["y_gold"]]
    
    y_pred_intervention = None
    if intervention:
        y_pred_intervention = sample[mapping["y_pred_intervention"]]        

    return y_pred_origin, mediator, y_gold, y_pred_intervention

def format_sample_for_eval(sample, dataset_name):
    
    if dataset_name == "ricechem":
        args = {"rubric": sample["mediator_rubric"]}
        meta = {"task_idx": sample["task_idx"]}
    
    elif dataset_name == "averitec":
        args = {"rubric": sample["mediator_rubric"]}
        meta = {
            "gold_rubric": sample["gold_rubric"],
            "gold_target": sample["gold_target"]
        }
    
    elif dataset_name == "tabfact":
        args = {"query": sample["mediator_query"]}
        meta = {"table_html_csv": sample["table_html_csv"]}
    
    else:
        raise ValueError(f"Unsupported dataset {dataset_name}")
    
    return args, meta



def calculate_id_faith(sample, dataset_name):
    
    y_pred_origin, _, _, _ = extract_values_from_dataset_sample(sample, dataset_name, intervention=False)
    
    args, meta = format_sample_for_eval(sample, dataset_name)
    dataset_tool = DATASET_TO_TOOL[dataset_name]

    tool_m_predicted = dataset_tool(args, meta)
    
    match = (tool_m_predicted == y_pred_origin)
        
    return match

def calculate_id_ooId_faith(sample, dataset_name, return_ood=False):
    
    
    origin_match = calculate_id_faith(sample, dataset_name)
    
    
    correction_type = "Local Edits" if len(sample['structure_intervention']['Local Edits']) > 0 else "Correction"
    # just take first element here for unification for LEC
    element_idx = random.randint(0, len(sample['structure_intervention']['Local Edits'])-1) if correction_type == "Local Edits" else 0
    intervention_sample = sample['structure_intervention'][correction_type][element_idx]
    
    _, _, _, y_pred_intervention = extract_values_from_dataset_sample(intervention_sample, dataset_name, intervention=True)

    args, meta = format_sample_for_eval(intervention_sample, dataset_name)
    dataset_tool = DATASET_TO_TOOL[dataset_name]
    tool_m_predicted = dataset_tool(args, meta)
    
    
    intervention_match = (tool_m_predicted == y_pred_intervention)
    result = origin_match & intervention_match
    
    if not return_ood:
        return result
    else:
        return result, intervention_match
    

def test():
    model_name = 'qwen3-8B'
    input_sample = all_prediction['ricechem']['standard'][model_name][0]['result'][2]
    
    assert calculate_id_faith(input_sample, 'ricechem') == True
    assert calculate_id_ooId_faith(input_sample, 'ricechem') == False
    print("Faith ID: ", calculate_id_faith(input_sample, 'ricechem'))
    print("faith Strong: ", calculate_id_ooId_faith(input_sample, 'ricechem'))
    print()
    
    input_sample = all_prediction['averitec']['standard'][model_name][0]['result'][-45]
    assert calculate_id_faith(input_sample, 'averitec') == True
    assert calculate_id_ooId_faith(input_sample, 'averitec') == True
    print("Faith ID: ", calculate_id_faith(input_sample, 'averitec'))
    print("faith Strong: ", calculate_id_ooId_faith(input_sample, 'averitec'))
    print()
    
    input_sample = all_prediction['tabfact']['standard'][model_name][0]['result'][0]
    assert calculate_id_faith(input_sample, 'tabfact') == False 
    assert calculate_id_ooId_faith(input_sample, 'tabfact') == False
    print("Faith ID: ", calculate_id_faith(input_sample, 'tabfact'))
    print("faith Strong: ", calculate_id_ooId_faith(input_sample, 'tabfact'))
    
    
test()

In [ ]:
DATASET_TOOL_KEY_MAP = {
    "ricechem": {
        "mediator": "mediator_rubric",
        "tool_pred_origin": "tool_rubric",
        "tool_pred_intervention": "tool_rubric_after_intervention"
    },
    "averitec": {
        "mediator": "mediator_rubric",
        "tool_pred_origin": "tool_rubric",
        "tool_pred_intervention": "tool_rubric_after_intervention"
    },
    "tabfact": {
        "mediator": "mediator_query",
        "tool_pred_origin": "tool_query",
        "tool_pred_intervention": "tool_query_after_intervention"
    }
}

def extract_values_from_dataset_sample_tool(sample, dataset_name, intervention=False):

    mapping = DATASET_TOOL_KEY_MAP[dataset_name]

    mediator = sample[mapping["mediator"]]
    tool_pred_origin = sample[mapping["tool_pred_origin"]]

    tool_pred_intervention = None
    if intervention:
        tool_pred_intervention = sample[mapping["tool_pred_intervention"]]

    return mediator, tool_pred_origin, tool_pred_intervention

def calculate_id_faith_tool(sample, dataset_name):
    
    
    mediator, tool_pred_origin, _ = extract_values_from_dataset_sample_tool(
        sample, dataset_name, intervention=False
    )
    
    args, meta = format_sample_for_eval(sample, dataset_name)
    dataset_tool = DATASET_TO_TOOL[dataset_name]

    # tool prediction from mediator
    tool_m_predicted = dataset_tool(args, meta)
    
    if dataset_name in ["ricechem", "averitec"]:
        tool_args = {"rubric": tool_pred_origin}
    elif dataset_name in ["tabfact"]:
        tool_args = tool_pred_origin
        
    tool_y_predicted = dataset_tool(tool_args, meta)
    
    match = (tool_m_predicted == tool_y_predicted)
    return match

def calculate_id_ooId_faith_tool(sample, dataset_name, return_ood=False):

    origin_match = calculate_id_faith_tool(sample, dataset_name)

    correction_type = (
        "Local Edits"
        if len(sample["structure_intervention"]["Local Edits"]) > 0
        else "Correction"
    )
    element_idx = (
        random.randint(0, len(sample["structure_intervention"]["Local Edits"]) - 1)
        if correction_type == "Local Edits"
        else 0
    )
    intervention_sample = sample["structure_intervention"][correction_type][element_idx]

    mediator, _, tool_pred_intervention = extract_values_from_dataset_sample_tool(
        intervention_sample, dataset_name, intervention=True
    )

    args, meta = format_sample_for_eval(intervention_sample, dataset_name)
    dataset_tool = DATASET_TO_TOOL[dataset_name]

    tool_m_predicted = dataset_tool(args, meta)

    if dataset_name in ["ricechem", "averitec"]:
        tool_args = {"rubric": tool_pred_intervention}
    elif dataset_name in ["tabfact"]:
        tool_args = tool_pred_intervention


    tool_y_predicted = dataset_tool(tool_args, meta)

    intervention_match = (tool_m_predicted == tool_y_predicted)

    if not return_ood:
        return origin_match & intervention_match
    else:
        return origin_match & intervention_match, intervention_match

        
        
def test():

    model_name = "qwen3-8B"
    idx = 48

    input_sample = all_prediction["ricechem"]["tool"][model_name][0]["result"][idx]

    if input_sample["generation_status"] != "error":

        print("Faith ID:", calculate_id_faith_tool(input_sample, "ricechem"))
        print("Faith Strong:", calculate_id_ooId_faith_tool(input_sample, "ricechem"))

    else:
        print("Error in generation")
    
    input_sample = all_prediction["averitec"]["tool"][model_name][0]["result"][idx]

    if input_sample["generation_status"] != "error":

        print("Faith ID:", calculate_id_faith_tool(input_sample, "averitec"))
        print("Faith Strong:", calculate_id_ooId_faith_tool(input_sample, "averitec"))

    else:
        print("Error in generation")
        
    input_sample = all_prediction["tabfact"]["tool"][model_name][0]["result"][34]
    if input_sample["generation_status"] != "error":

        print("Faith ID:", calculate_id_faith_tool(input_sample, "tabfact"))
        print("Faith Strong:", calculate_id_ooId_faith_tool(input_sample, "tabfact"))

    else:
        print("Error in generation")
        
test()

In [ ]:
dataset_names = []
run_type_list = []
model_name_list = []
intervention_type_list = []
id_faith_list = []
id_ooId_faith_list = []
f_ood_list = []

nice_looking_dataset_name = {"ricechem": "RiceChem",
                            "averitec": "AVeriTeC",
                            "tabfact": "TabFact"}

nice_looking_name_map = {
    "llama32-1B": "Llama-3.2 1B",
    "qwen3-1.7B": "Qwen-3 1.7B",
    "gemma2-2B":   "Gemma-2 2B",
    "falcon3-3B":  "Falcon-3 3B",
    "llama32-3B":  "Llama-3.2 3B",
    "qwen3-4B":    "Qwen-3 4B",
    "falcon3-7B":  "Falcon-3 7B",
    "qwen3-8B":    "Qwen-3 8B",
    "llama31-8B":  "Llama-3.1 8B",
    "qwen3-14B": "Qwen-3 14B",
    "qwen3-32B": "Qwen-3 32B",
    "llama31-70B": "Llama-3.1 70B",
    "qwen3-235B-a22B": "Qwen3-235B-A22B"
}

total_items_tqdm = 0
for ds_name in dataset_name_list:
    total_items_tqdm += len(run_types_list) * len(models_name) * len(all_prediction[ds_name]['standard']["qwen3-8B"][0]['result'])
    
    
def filter_samples_heuristic(sample, dataset_name):
    generation_status = sample['generation_status']
    if generation_status == 'error':
        return 'error'
    else:
        if dataset_name == 'averitec':
            if sample['generation_status'] == 'incorrect' and len(sample['gold_rubric']) != len(sample['mediator_rubric']):
                return 'error'
        else:
            return generation_status
            

with tqdm(total=total_items_tqdm, desc="Processing") as pbar:
    for dataset in dataset_name_list:
        for run_type in run_types_list:
            for model in models_name:
                if run_type in all_prediction[dataset]:
                    results = all_prediction[dataset][run_type][model][0]['result']
                    for idx, sample in enumerate(results):
                        generation_status = filter_samples_heuristic(sample, dataset)
                        if generation_status != 'error':
                            intervention_type = "counterfact" if len(sample['structure_intervention']['Local Edits']) > 0 else "correction"
                            if run_type == 'tool':
                                id_faith = calculate_id_faith_tool(sample, dataset)
                                id_ooId_faith, f_ood = calculate_id_ooId_faith_tool(sample, dataset, return_ood=True)
                            else:
                                id_faith = calculate_id_faith(sample, dataset)
                                id_ooId_faith, f_ood = calculate_id_ooId_faith(sample, dataset, return_ood=True)
                        else:
                            intervention_type = 'no_intervention'
                            id_faith, id_ooId_faith, f_ood = False, False, False


                        dataset_names.append(nice_looking_dataset_name[dataset])
                        run_type_list.append(run_type)
                        model_name_list.append(nice_looking_name_map[model])
                        intervention_type_list.append(intervention_type)
                        id_faith_list.append(id_faith)
                        id_ooId_faith_list.append(id_ooId_faith)
                        f_ood_list.append(f_ood)

                        pbar.update(1)
                

df_exps = pd.DataFrame()
df_exps['dataset_name'] = dataset_names
df_exps['run_type_list'] = run_type_list
df_exps['model_name_list'] = model_name_list
df_exps['intervention_type_list'] = intervention_type_list
df_exps['id_faith_list'] = id_faith_list
df_exps['id_ooId_faith_list'] = id_ooId_faith_list
df_exps['f_ood_list'] = f_ood_list

# exclude it
df_exps = df_exps[df_exps['model_name_list'] != 'Llama-3.2 1B']

df_exps.head()

In [ ]:
df_exps[(df_exps['run_type_list'] == 'tool') & (df_exps['model_name_list'] == 'Falcon-3 3B')]['id_ooId_faith_list'].value_counts()

### Plot Vars

In [ ]:
MODEL_ORDER = ['Qwen-3 1.7B', 'Gemma-2 2B', 'Falcon-3 3B', 'Llama-3.2 3B', 'Qwen-3 4B', 'Falcon-3 7B', 'Qwen-3 8B', 'Llama-3.1 8B', "Qwen-3 14B", "Qwen-3 32B", "Llama-3.1 70B", "Qwen3-235B-A22B"]
DATASET_ORDER = ['RiceChem', 'AVeriTeC', 'TabFact']

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 1.1,
    "grid.color": "#d9d9d9",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "xtick.color": "#333333",
    "ytick.color": "#333333"
})

# Define a professional color palette (distinct colors for models)
model_colors = {
    # "Llama-3.2 1B": "#1f77b4",
    "Qwen-3 1.7B": "#ff7f0e",
    "Gemma-2 2B": "#2ca02c",
    "Falcon-3 3B": "#d62728",
    "Llama-3.2 3B": "#9467bd",
    "Qwen-3 4B": "#8c564b",
    "Falcon-3 7B": "#e377c2",
    "Qwen-3 8B": "#7f7f7f",
    "Llama-3.1 8B": "#bcbd22",
    "Qwen-3 14B": "#8c842b",
    "Qwen-3 32B": "#8c120b",
    "Llama-3.1 70B": "#bcbd44",
    "Qwen3-235B-A22B": "#8c564b"
}

XLABEL_SIZE = 11
YLABEL_SIZE = 11
TITLE_SIZE = 10
LEGEND_FONT = 8
TICKS_SIZE = 10

### RQ1 table

In [ ]:
import pandas as pd
import numpy as np

# select stadart run for first table
df_standard = df_exps[df_exps['run_type_list'] == 'standard'].copy()

df_standard['id_faith_list'] = pd.to_numeric(df_standard['id_faith_list'], errors='raise')
df_standard['id_ooId_faith_list'] = pd.to_numeric(df_standard['id_ooId_faith_list'], errors='raise')
df_standard['f_ood_list'] = pd.to_numeric(df_standard['f_ood_list'], errors='raise')

def format_mu_std(mean):
    # if pd.isna(std):
    return f"{mean:.2f}"
     # Use Unicode ± for display, or \\pm if exporting to LaTeX
    # return f"{mean:.2f} ± {std:.2f}"

models = MODEL_ORDER
datasets = DATASET_ORDER

data = {}
for model in models:
    for dataset in datasets:
        # Filter for this model-dataset pair
        mask = (df_standard['model_name_list'] == model) & (df_standard['dataset_name'] == dataset)
        subset = df_standard[mask]
        
        # Calculate ID stats (from id_faith_list)
        id_mean = round(subset['id_faith_list'].mean(), 2)
        data[(dataset, 'ID')] = data.get((dataset, 'ID'), []) + [id_mean]

        # Calculate OOD stats (from f_ood_list)
        ood_mean = round(subset['f_ood_list'].mean(), 2)
        data[(dataset, 'OOD')] = data.get((dataset, 'OOD'), []) + [ood_mean]
        
        
        # Calculate OO(I)D stats (from id_ooId_faith_list)
        ooid_mean = round(subset['id_ooId_faith_list'].mean(), 2)
        data[(dataset, 'OO(I)D')] = data.get((dataset, 'OO(I)D'), []) + [ooid_mean]

        data[(dataset, 'Delta')] = data.get((dataset, 'Delta'), []) + [id_mean - ooid_mean]

final_table = pd.DataFrame(data, index=models)
final_table.index.name = 'model_name_list'


# Display in notebook
display(final_table)

In [ ]:
final_table.to_markdown()

### RQ2 Symmetry plot Tables

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

datasets = DATASET_ORDER
models = MODEL_ORDER

n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(4 * n_datasets, 4))

if n_datasets == 1:
    axes = [axes]

for idx, dataset in enumerate(datasets):
    ax = axes[idx]
    
    df_subset = df_standard[df_standard['dataset_name'] == dataset]
    
    summary_data = []
    for model in models:
        df_model = df_subset[df_subset['model_name_list'] == model]
        
        correction_df = df_model[df_model['intervention_type_list'] == 'correction']
        counterfact_df = df_model[df_model['intervention_type_list'] == 'counterfact']
        
        correction_mean = correction_df['id_ooId_faith_list'].mean()
        counterfact_mean = counterfact_df['id_ooId_faith_list'].mean()
        
        if ((len(correction_df) > 10) and (len(counterfact_df) > 10)):
            summary_data.append({
                'model': model,
                'correction': correction_mean,
                'counterfact': counterfact_mean
            })
    
    summary_df = pd.DataFrame(summary_data)
    
    ax.plot([0, 1], [0, 1],
            linestyle='--',
            linewidth=1.3,
            color="#555555",
            alpha=0.6,
            zorder=1,
            label='Symmetry')
    
    # Scatter plot for each model
    for _, row in summary_df.iterrows():
        ax.scatter(row['correction'], row['counterfact'], 
                  s=110,
                  color=model_colors[row['model']], 
                  edgecolors='black',  # Changed from 'white' to 'black'
                  linewidth=0.5,
                  alpha=0.95,
                  zorder=5)
        
        # Add model label with slight offset to avoid overlapping the point
        ax.annotate(row['model'], 
                   xy=(row['correction'], row['counterfact']),
                   xytext=(10, 5),
                   textcoords='offset points',
                   fontsize=LEGEND_FONT,
                   color="#222222",
                   alpha=0.9)
    
    # Styling for each subplot
    ax.set_xlim(-0.02, 1)
    ax.set_ylim(-0.02, 1)
    
    ax.set_xlabel('Correction', fontsize=XLABEL_SIZE)
    ax.set_ylabel('Counterfact', fontsize=YLABEL_SIZE)
    ax.set_title(dataset, fontsize=TITLE_SIZE)
    ax.tick_params(axis='x', labelsize=TICKS_SIZE)  # controls x-tick label size
    ax.tick_params(axis='y', labelsize=TICKS_SIZE)  # controls x-tick label size

    
    # Grid styling
    ax.grid(True)
    ax.set_axisbelow(True)

    # Clean spines
    sns.despine(ax=ax)

# Create horizontal legend below all subplots
legend_handles = []
for model, color in model_colors.items():
    legend_handles.append(plt.Line2D([0], [0], marker='o', color='w', 
                                      markerfacecolor=color, 
                                      markersize=10, 
                                      markeredgecolor='black', 
                                      markeredgewidth=0.5,
                                      label=model))

# Add legend below the figure
fig.legend(handles=legend_handles, 
           loc='lower center', 
           bbox_to_anchor=(0.5, 0), 
           ncol=5, 
           frameon=False, 
           fontsize=LEGEND_FONT,
           columnspacing=1.5,
           handletextpad=0.5)


plt.tight_layout(rect=[0, 0.1, 1, 1])  # Adjust layout to make room for legend
plt.savefig('symmetry_analysis.png', bbox_inches='tight', dpi=300)
plt.show()

### R3 Tool-use

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch

# =============================================================================
# STEP 0: Define model order from your reference dataframe
# =============================================================================

model_order = MODEL_ORDER
print(f"Model order: {list(model_order)}")

# =============================================================================
# STEP 1: Prepare the data - CORRECT DELTA CALCULATION
# =============================================================================

# Group by dataset, model, run_type
grouped = df_exps.groupby(['dataset_name', 'model_name_list', 'run_type_list'])

# Compute mean of each metric separately, THEN compute delta
def compute_delta(group):
    id_mean = group['id_faith_list'].mean()
    ooid_mean = group['id_ooId_faith_list'].mean()
    return pd.Series({
        'id_mean': id_mean,
        'ooid_mean': ooid_mean,
        'delta': id_mean - ooid_mean  # ✅ Difference of means
    })

aggregated = grouped.apply(compute_delta).reset_index()

# Pivot to wide format: one row per (dataset, model), columns for 'standard' and 'tool'
pivot_df = aggregated.pivot_table(
    index=['dataset_name', 'model_name_list'], 
    columns='run_type_list', 
    values='delta', 
    aggfunc='first'  # Already aggregated, just reshape
).reset_index()

# Ensure both run types exist as columns
for col in ['standard', 'tool']:
    if col not in pivot_df.columns:
        pivot_df[col] = np.nan

# Get unique datasets
datasets = pivot_df['dataset_name'].unique()
print(f"Found datasets: {list(datasets)}")

# =============================================================================
# STEP 2: Create the visualization
# =============================================================================

n_datasets = len(datasets)
fig, axes = plt.subplots(1, n_datasets, figsize=(5 * n_datasets, 4))
if n_datasets == 1:
    axes = [axes]

# Styling parameters
transparent_alpha = 0.25
opaque_alpha = 1.0
bar_width = 0.8
arrow_color = 'forestgreen'
text_color = 'forestgreen'

datasets = DATASET_ORDER
all_datasets_list = []
for idx, (ax, dataset) in enumerate(zip(axes, datasets)):
    # Filter data for current dataset
    data = pivot_df[pivot_df['dataset_name'] == dataset].copy()
    
    # Filter AND ORDER models according to model_order
    data = data[data['model_name_list'].isin(model_order)]
    data['model_name_list'] = pd.Categorical(
        data['model_name_list'], 
        categories=model_order, 
        ordered=True
    )
    data = data.sort_values('model_name_list')
    
    data = data.dropna(subset=['standard', 'tool'], how='all')
    all_datasets_list.append(data)
    
    if len(data) == 0:
        ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', 
               transform=ax.transAxes, fontsize=12, style='italic')
        ax.set_title(f'Dataset: {dataset}', fontsize=14, fontweight='bold', pad=20)
        continue
    
    # Prepare plot positions
    models = data['model_name_list'].tolist()
    x_pos = np.arange(len(models))
    
    standard_vals = data['standard'].values
    tool_vals = data['tool'].values
    
    # =========================================================================
    # Plot nested VERTICAL bars with MODEL-SPECIFIC COLORS
    # =========================================================================
    for i, model in enumerate(models):
        color = model_colors.get(model, '#999999')
        
        # Transparent wider bar = standard (baseline)
        ax.bar(x_pos[i], standard_vals[i], width=bar_width, color=color, 
                alpha=transparent_alpha, zorder=2, edgecolor='black', linewidth=0.5)
        
        # Opaque narrower bar = tool (intervention)
        ax.bar(x_pos[i], tool_vals[i], width=bar_width * 0.7, color=color, 
                alpha=opaque_alpha, zorder=2, edgecolor='black', linewidth=0.5)
    
    # =========================================================================
    # Add green connectors + delta text showing change from standard → tool
    # =========================================================================
    for i, (std, tool) in enumerate(zip(standard_vals, tool_vals)):
        if pd.isna(std) or pd.isna(tool):
            continue
            
        delta_change = tool - std
        delta_rounded = round(delta_change, 2)
        
        # # Arrow/line position: slightly offset from bar center
        # x_arrow = x_pos[i] + bar_width * 0.12
        x_arrow = x_pos[i]
        
        # ---- CHANGE 1: draw line instead of arrow if |delta| < 0.05 ----
        if abs(delta_change) < 0.05:
            # Draw a simple vertical line (no arrowhead)
            ax.plot([x_arrow, x_arrow], [std, tool], 
                    color=arrow_color, linewidth=2.5, zorder=3)
        else:
            arrow = FancyArrowPatch(
                (x_arrow, std), (x_arrow, tool),
                color=arrow_color, 
                arrowstyle='->', 
                linewidth=2.0, 
                mutation_scale=18,
                zorder=3
            )
            ax.add_patch(arrow)
        
        # ---- CHANGE 3: place delta text exactly above bars (centered) ----
        text_offset = 0.05
        text_y = max(std, tool) + text_offset
        # Use the bar's center x_pos[i] for exact horizontal centering
        text_x = x_pos[i]
        
        ax.text(text_x, text_y, f'{delta_rounded:+.2f}', 
               color=text_color, va='bottom', ha='center', fontsize=9, 
               fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', 
                                           facecolor='white', 
                                           edgecolor=arrow_color, 
                                           alpha=0.85),
               zorder=4)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(models, fontsize=TICKS_SIZE, rotation=45, ha='right')
    ax.tick_params(axis='y', labelsize=TICKS_SIZE)
    ax.set_title(dataset, fontsize=TITLE_SIZE)
    
    # ---- CHANGE 2: ylabel only on leftmost subplot, horizontal, no bold ----
    if idx == 0:
        ax.set_ylabel('Delta', fontsize=YLABEL_SIZE, fontweight='normal', labelpad=10)
    
    ax.axhline(0, color='gray', linewidth=0.6, alpha=0.4, zorder=0)  # zero reference line
    
    ax.grid(True)
    ax.set_ylim(-0.02, 0.75)
    ax.set_axisbelow(True)

    # Clean spines
    sns.despine(ax=ax)

plt.savefig('tool_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
all_df_list_for_tool = pd.concat(all_datasets_list, axis=0)
# all_df_list_for_tool
all_df_list_for_tool = all_df_list_for_tool[['dataset_name', 'model_name_list', 'standard', 'tool']].round(2)
all_df_list_for_tool['delta'] = all_df_list_for_tool['standard'] - all_df_list_for_tool['tool']

In [ ]:
all_df_list_for_tool.groupby(['dataset_name'])[['standard', 'tool', 'delta']].mean().to_markdown()

### R4 Prompt

In [ ]:
# 1. Create the pivot table
table = df_exps.pivot_table(
    index='model_name_list', 
    columns=['dataset_name', 'run_type_list'], 
    values='id_ooId_faith_list', 
    aggfunc='mean'
).round(3)

# 2. Define the desired order for prompt types (run_type_list)
prompt_order = ['standard', 'detailed', 'max_detailed']  # Adjust if your actual strings differ (e.g., 'Standard')

# 3. Reorder columns: for each dataset, sort sub-columns by prompt_order
# Get unique datasets
datasets = DATASET_ORDER

# 3. Reorder rows (models) according to MODEL_ORDER
# Filter to only include models that exist in the table
valid_models = MODEL_ORDER
table = table.reindex(index=valid_models)

# Build new column order list
new_columns = []
for ds in datasets:
    for prompt in prompt_order:
        # Check if this (dataset, prompt) combination exists in columns
        if (ds, prompt) in table.columns:
            new_columns.append((ds, prompt))

# Reindex columns to enforce order
table = table.reindex(columns=pd.MultiIndex.from_tuples(new_columns, names=['dataset_name', 'run_type_list']))
table = table.round(2)
# 4. Optional: Sort models alphabetically or by size (customize as needed)
# table = table.sort_index() 

# 5. Display
display(table)

In [ ]:
# 1. Group by prompt type and dataset, calculate mean and std across all models
summary = df_exps.groupby(['run_type_list', 'dataset_name'])['id_ooId_faith_list'].agg(['mean', 'std']).round(3)

# 2. Format as "mean ± std"
summary['value'] = summary.apply(lambda row: f"{round(row['mean'], 2)} ± {round(row['std'], 2)}", axis=1)

# 3. Pivot to get prompt types as rows, datasets as columns
table_small = summary['value'].unstack(level='dataset_name')

# 4. Reorder rows and columns to match your preferred order
PROMPT_ORDER = ['standard', 'detailed', 'max_detailed']  # Adjust case if needed
DATASET_ORDER = DATASET_ORDER  # Or use nice names

# Filter to existing indices/columns and reindex
table_small = table_small.reindex(
    index=[p for p in PROMPT_ORDER if p in table_small.index],
    columns=[d for d in DATASET_ORDER if d in table_small.columns]
)

# 6. Display
display(table_small)